In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, "../utils/")
from data_loader import load_data
from sklearn.model_selection import train_test_split
import random

In [2]:
def scale_dataframe(df: pd.DataFrame):
    """
    Scales the gene effect data columns of a DataFrame to a 0-1 range.
    The first column (ID) and the last two columns (age and sex) are not scaled.

    Parameters:
    df (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: The scaled DataFrame.
    """
    col_num = df.shape[1]
    df_to_scale = df.iloc[:, 1:col_num-1]
    
    scaler = MinMaxScaler(feature_range=(0,1))
    scaled_df = scaler.fit_transform(df_to_scale)
    
    scaled_df = pd.DataFrame(scaled_df)
    scaled_df.insert(0, df.columns[0], df[df.columns[0]])
    scaled_df.insert(col_num-1, df.columns[col_num-1], df[df.columns[col_num-1]])
    scaled_df.columns = df.columns
    
    return scaled_df

In [3]:
def save_dataframe(df, file_path: pathlib.Path):
    """
    Saves a DataFrame to a specified file path.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    file_path (str): The file path to save the DataFrame.
    """
    df = df.reset_index(drop=True)
    df.to_parquet(file_path, index=False)
    print(f"DataFrame saved to {file_path}. Shape: {df.shape}")
    print(df.head(3))

In [4]:
random.seed(18)
print(random.random())

0.18126486333322134


In [5]:
# load all of the data
data_directory = "../0.data-download/data/"
model_df, effect_df = load_data(data_directory, adult_or_pediatric="all")

/home/gway/repos/gene_dependency_representations/1.data-exploration/../utils/data_loader.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  effect_df = effect_df.set_index(id_column).sort_index(ascending=True).reset_index()


/home/gway/repos/gene_dependency_representations/1.data-exploration/../utils/data_loader.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  effect_df = effect_df.set_index(id_column).reindex(index=list(mod_vs_eff_ids)).reset_index()


In [6]:
# verifying that the ModelIDs in model_df and effect_df are alligned
model_df["ID_allignment_verify"] = np.where(
    effect_df["ModelID"] == model_df["ModelID"], "True", "False"
)
verrify = len(model_df["ID_allignment_verify"].unique())
print(model_df["ID_allignment_verify"])
print(
    f"There is {verrify} output object contained in the ID_allignment_verify column \n"
)

0       True
1       True
2       True
3       True
4       True
        ... 
1145    True
1146    True
1147    True
1148    True
1149    True
Name: ID_allignment_verify, Length: 1150, dtype: str
There is 1 output object contained in the ID_allignment_verify column 



In [7]:
# assign 'AgeCategory' and 'Sex' columns to the effect dataframe as a single column
presplit_effect_df = effect_df.assign(
    age_and_sex=model_df.AgeCategory.astype(str) + "_" + model_df.Sex.astype(str)
)
presplit_effect_df

/tmp/ipykernel_933445/3051282021.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  presplit_effect_df = effect_df.assign(


,ModelID,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),...,ZWINT (11130),ZXDA (7789),ZXDB (158586),ZXDC (79364),ZYG11A (440590),ZYG11B (79699),ZYX (7791),ZZEF1 (23140),ZZZ3 (26009),age_and_sex
0,ACH-000947,-0.098552,-0.016738,0.044657,-0.013973,-0.051237,-0.087004,-0.036820,-0.126643,-0.081623,...,-0.782085,0.173384,0.063774,0.000597,-0.074666,-0.115861,-0.070837,-0.180006,-0.089104,Adult_Female
1,ACH-000271,-0.371638,0.084831,-0.043180,0.021181,-0.206486,-0.099801,0.137065,-0.333629,-0.162360,...,0.309595,0.142709,-0.032421,-0.002199,-0.057900,0.063855,0.026486,-0.038851,0.054204,Adult_Male
2,ACH-000483,0.044141,-0.075275,0.024215,0.186595,-0.180126,-0.101808,0.013707,-0.103095,0.218965,...,-0.458373,0.178651,0.040413,-0.000327,-0.012874,-0.213376,-0.275771,-0.100323,-0.385761,Adult_Male
3,ACH-001970,-0.222450,-0.042154,0.027603,0.025462,-0.052482,-0.080641,0.047727,-0.299124,-0.061428,...,-0.577212,-0.067261,-0.186247,-0.285479,-0.397367,0.066237,0.048924,0.098406,-0.247333,Adult_Male
4,ACH-000212,-0.286788,-0.037546,-0.033607,0.212746,-0.255601,-0.285290,-0.038612,0.065101,0.090748,...,-0.125246,0.111643,0.096212,0.067912,0.044364,-0.205857,0.017369,-0.128301,-0.160112,Adult_Female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,ACH-002925,-0.158397,-0.036294,0.094842,-0.043724,-0.111456,0.083429,0.116124,-0.140481,-0.027413,...,-0.776332,-0.036467,0.109655,-0.215080,-0.004304,-0.247449,-0.222342,-0.079772,-0.196377,Unknown_Male
1146,ACH-000350,-0.091023,-0.098555,0.012734,0.097472,-0.238102,-0.097257,0.043564,0.055570,-0.021067,...,-0.380785,0.154875,0.232575,0.129108,-0.072800,-0.240959,0.093093,-0.095402,-0.072806,Adult_Male
1147,ACH-000740,-0.124825,-0.295578,-0.007369,0.068595,-0.152515,-0.033633,0.097236,0.156819,0.037758,...,-0.813277,0.011322,0.009062,0.008307,0.088289,-0.146350,0.052067,-0.036722,-0.322246,Adult_Male
1148,ACH-000078,-0.228053,0.043147,0.080473,0.043626,0.081738,-0.163086,0.069353,-0.103859,-0.041300,...,-0.167761,0.093288,0.240991,0.025793,0.011512,-0.743012,-0.040440,-0.002424,-0.327981,Pediatric_Male


In [8]:
groups = model_df.groupby("AgeCategory")
df_list = []
for name, df in groups:

    # only looking for samples that contain Adult or Pediatric information
    if name == "Adult" or name == "Pediatric":
        df_list.append(df)

# merge sample dataframes through concatentation and reorganize so that ModelIDs are in alphabetical order
new_df = pd.concat(df_list, axis=0)
new_df = new_df.set_index("ModelID")
new_df = new_df.sort_index(ascending=True)
new_df = new_df.reset_index()

In [9]:
# creating a list of ModelIDs that correlate to pediatric and adult samples
PA_effect_IDs = new_df["ModelID"].tolist()

PA_IDs = set(PA_effect_IDs) & set(presplit_effect_df["ModelID"].tolist())

# creating a new gene effect data frame containing correlating ModelIDs to the filtered sample info IDs
PA_effect_df = presplit_effect_df.loc[
    presplit_effect_df["ModelID"].isin(PA_IDs)
].reset_index(drop=True)

In [10]:
# split the data based on age category and sex
train_df, testandvalidation_df = train_test_split(
    PA_effect_df, test_size=0.3, stratify=PA_effect_df.age_and_sex
)
train_df.reset_index(drop=True,inplace=True)
testandvalidation_df.reset_index(drop=True,inplace=True)
test_df, val_df = train_test_split(
    testandvalidation_df, test_size=0.5
)
test_df.reset_index(drop=True,inplace=True)
val_df.reset_index(drop=True,inplace=True)

In [11]:
#save each dataframe
save_dataframe(train_df, pathlib.Path("data/VAE_train_df.parquet").resolve())
save_dataframe(test_df, pathlib.Path("data/VAE_test_df.parquet").resolve())
save_dataframe(val_df, pathlib.Path("data/VAE_val_df.parquet").resolve())

DataFrame saved to /home/gway/repos/gene_dependency_representations/1.data-exploration/data/VAE_train_df.parquet. Shape: (670, 17109)


      ModelID  A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)  \
0  ACH-000213 -0.137704     -0.209328  0.035861        0.095985   
1  ACH-002511 -0.058905     -0.018946  0.067556        0.159318   
2  ACH-001328 -0.085910     -0.044116  0.025245       -0.127086   

   A3GALT2 (127550)  A4GALT (53947)  A4GNT (51146)  AAAS (8086)  AACS (65985)  \
0         -0.103421       -0.038196      -0.113301    -0.082544     -0.172621   
1         -0.170766        0.021027      -0.001709     0.022658     -0.070779   
2         -0.003605       -0.113595       0.043094     0.011756     -0.029211   

   ...  ZWINT (11130)  ZXDA (7789)  ZXDB (158586)  ZXDC (79364)  \
0  ...      -0.518846     0.089815       0.182594      0.013440   
1  ...      -0.128634    -0.038578       0.017898     -0.101944   
2  ...      -0.528823     0.067074       0.026188     -0.112285   

   ZYG11A (440590)  ZYG11B (79699)  ZYX (7791)  ZZEF1 (23140)  ZZZ3 (26009)  \
0         0.284268       -0.325811    0.028021      -0.013

DataFrame saved to /home/gway/repos/gene_dependency_representations/1.data-exploration/data/VAE_test_df.parquet. Shape: (144, 17109)
      ModelID  A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)  \
0  ACH-000714 -0.210799      0.091533  0.076634        0.055004   
1  ACH-001053 -0.027691     -0.103021 -0.017925        0.080065   
2  ACH-001652  0.026694     -0.149711 -0.038788        0.074027   

   A3GALT2 (127550)  A4GALT (53947)  A4GNT (51146)  AAAS (8086)  AACS (65985)  \
0         -0.066242       -0.167319       0.019762    -0.366983      0.060447   
1         -0.084883       -0.289116       0.121507     0.017827      0.158713   
2         -0.072840       -0.058141       0.082337    -0.088134      0.043527   

   ...  ZWINT (11130)  ZXDA (7789)  ZXDB (158586)  ZXDC (79364)  \
0  ...      -0.298716     0.104249      -0.035522     -0.086779   
1  ...      -0.598039     0.037359       0.078349      0.049545   
2  ...      -0.510516     0.085163       0.112050     -0.094280   

   Z

DataFrame saved to /home/gway/repos/gene_dependency_representations/1.data-exploration/data/VAE_val_df.parquet. Shape: (144, 17109)


      ModelID  A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)  \
0  ACH-001977 -0.142612     -0.214454  0.106836        0.062783   
1  ACH-000834 -0.045891     -0.260131  0.009378       -0.010856   
2  ACH-001367 -0.134936     -0.051623 -0.094036        0.016176   

   A3GALT2 (127550)  A4GALT (53947)  A4GNT (51146)  AAAS (8086)  AACS (65985)  \
0         -0.170634       -0.097923      -0.020253    -0.117966     -0.128151   
1         -0.036821        0.072006       0.046686     0.048228     -0.099161   
2         -0.112589       -0.264421       0.183960    -0.247183     -0.117249   

   ...  ZWINT (11130)  ZXDA (7789)  ZXDB (158586)  ZXDC (79364)  \
0  ...      -0.452444    -0.018101       0.028648     -0.376386   
1  ...      -0.490826     0.158543       0.154925     -0.151508   
2  ...      -0.283187     0.161722       0.126249     -0.166443   

   ZYG11A (440590)  ZYG11B (79699)  ZYX (7791)  ZZEF1 (23140)  ZZZ3 (26009)  \
0        -0.055938       -0.112256   -0.066711      -0.135

In [12]:
# create a data frame of both test and train gene effect data with sex, AgeCategory, and ModelID for use in later t-tests
# load in the data

# create dataframe containing the genes that passed an initial QC (see Pan et al. 2022) and a saturated signal qc, then extracting their corresponding gene label
gene_dict_df = pd.read_parquet("../0.data-download/data/CRISPR_gene_dictionary.parquet")
gene_list_passed_qc = gene_dict_df.loc[gene_dict_df["qc_pass"], 'dependency_column'].tolist()
concat_frames = [train_df, test_df, val_df]
train_and_test = pd.concat(concat_frames).reset_index(drop=True)
train_and_test[["AgeCategory", "Sex"]] = train_and_test.age_and_sex.str.split(
    pat="_", expand=True
)
train_and_test_subbed = train_and_test.filter(gene_list_passed_qc, axis=1)
metadata_holder = pd.DataFrame()
metadata = metadata_holder.assign(
    ModelID=train_and_test.ModelID.astype(str),
    AgeCategory=train_and_test.AgeCategory.astype(str),
    Sex=train_and_test.Sex.astype(str),
)

metadata_df_dir = pathlib.Path("data/metadata_df.parquet").resolve()
metadata.to_parquet(metadata_df_dir, index=False)

train_and_test_subbed_dir = pathlib.Path("data/train_and_test_subbed.parquet").resolve()
train_and_test_subbed.to_parquet(train_and_test_subbed_dir, index=False)

/tmp/ipykernel_933445/2181949518.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_and_test[["AgeCategory", "Sex"]] = train_and_test.age_and_sex.str.split(
/tmp/ipykernel_933445/2181949518.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_and_test[["AgeCategory", "Sex"]] = train_and_test.age_and_sex.str.split(
